### 1. Importaciones y configuración de rutas

In [20]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import os

# Configuración de rutas del proyecto
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = BASE_DIR / "data"
SRC_DIR = BASE_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from utils import modelo_logistico, ajustar_cinetica

os.makedirs(DATA_DIR, exist_ok=True)
print("✅ Entorno configurado correctamente.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Entorno configurado correctamente.


### 2. Generación automática de 10 aislados fúngicos

In [21]:
np.random.seed(42)
tiempo = np.array([0, 12, 24, 36, 48, 60, 72, 84, 96])

cepas = [
    "Pleurotus_ostreatus_A1", "Ganoderma_lucidum_G2", "Trametes_versicolor_T3",
    "Lentinula_edodes_L4", "Agaricus_bisporus_AB5", "Schizophyllum_commune_S6",
    "Grifola_frondosa_G7", "Hericium_erinaceus_H8", "Pycnoporus_sanguineus_P9",
    "Fomes_fomentarius_F10"
]

excel_path = DATA_DIR / "Datos_Fermentacion_Completo.xlsx"

# Creamos un escritor de Excel para guardar cada cepa en una hoja distinta
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    for cepa in cepas:
        K_rand = np.random.uniform(7.5, 13.0)
        mu_rand = np.random.uniform(0.08, 0.25)
        n0_rand = 0.1
        
        biomasa = modelo_logistico(tiempo, K_rand, mu_rand, n0_rand) + np.random.normal(0, 0.12, size=len(tiempo))
        biomasa = np.clip(biomasa, 0.05, None)
        
        df_cepa = pd.DataFrame({'tiempo': tiempo, 'biomasa': biomasa})
        df_cepa.to_excel(writer, sheet_name=cepa, index=False)

print(f"✅ Archivo único generado exitosamente en: {excel_path}")
print(f"Hojas integradas ({len(cepas)} cepas): {cepas}")

✅ Archivo único generado exitosamente en: C:\Users\ivanc\Desktop\2. Inversiones\Python_Clases\fungi-kinetic-analytics\data\Datos_Fermentacion_Completo.xlsx
Hojas integradas (10 cepas): ['Pleurotus_ostreatus_A1', 'Ganoderma_lucidum_G2', 'Trametes_versicolor_T3', 'Lentinula_edodes_L4', 'Agaricus_bisporus_AB5', 'Schizophyllum_commune_S6', 'Grifola_frondosa_G7', 'Hericium_erinaceus_H8', 'Pycnoporus_sanguineus_P9', 'Fomes_fomentarius_F10']


### 3. Gráfica interactiva con Plotly Express

In [22]:
hojas_dict = pd.read_excel(excel_path, sheet_name=None)
dfs_totales = []

for nombre_hoja, df_hoja in hojas_dict.items():
    df_temp = df_hoja.copy()
    df_temp.columns = [str(c).strip().lower() for c in df_temp.columns]
    df_temp['Aislamiento'] = nombre_hoja
    dfs_totales.append(df_temp)

df_global = pd.concat(dfs_totales, ignore_index=True)

fig = px.line(
    df_global, 
    x='tiempo', 
    y='biomasa', 
    color='Aislamiento',
    markers=True,
    title='<b>Análisis Exploratorio: Cinética de Crecimiento desde Archivo Único</b>',
    labels={'tiempo': 'Tiempo de Fermentación (h)', 'biomasa': 'Concentración de Biomasa (g/L)'}
)

fig.update_layout(
    template='plotly_white',
    title_font_size=16,
    hovermode='x unified',
    legend_title_text='<b>Aislados</b>'
)

fig.show()

### 4. Extracción de Parámetros Cinéticos (Ajuste No Lineal)

In [23]:
resultados = []

print("="*75)
print(f"{'REPORTE DE AJUSTE CINÉTICO DESDE EXCEL ÚNICO':^75}")
print("="*75 + "\n")

for nombre_hoja, df_hoja in hojas_dict.items():
    params = ajustar_cinetica(df_hoja)
    if params:
        params['Aislamiento'] = nombre_hoja
        resultados.append(params)
        
        print(f"Cepa: {nombre_hoja}")
        print(f"  • Tasa específica de crecimiento (μ): {params['mu']:.4f} h⁻¹")
        print(f"  • Capacidad de carga (K): {params['K']:.4f} g/L")
        print(f"  • Bondad de ajuste (R²): {params['R2']:.4f}")
        print("-" * 50)

               REPORTE DE AJUSTE CINÉTICO DESDE EXCEL ÚNICO                

Cepa: Pleurotus_ostreatus_A1
  • Tasa específica de crecimiento (μ): 0.2269 h⁻¹
  • Capacidad de carga (K): 9.6020 g/L
  • Bondad de ajuste (R²): 0.9996
--------------------------------------------------
Cepa: Ganoderma_lucidum_G2
  • Tasa específica de crecimiento (μ): 0.1734 h⁻¹
  • Capacidad de carga (K): 9.1433 g/L
  • Bondad de ajuste (R²): 0.9991
--------------------------------------------------
Cepa: Trametes_versicolor_T3
  • Tasa específica de crecimiento (μ): 0.1697 h⁻¹
  • Capacidad de carga (K): 8.6142 g/L
  • Bondad de ajuste (R²): 0.9993
--------------------------------------------------
Cepa: Lentinula_edodes_L4
  • Tasa específica de crecimiento (μ): 0.2225 h⁻¹
  • Capacidad de carga (K): 7.6525 g/L
  • Bondad de ajuste (R²): 0.9987
--------------------------------------------------
Cepa: Agaricus_bisporus_AB5
  • Tasa específica de crecimiento (μ): 0.1205 h⁻¹
  • Capacidad de carga (K): 7.943

### 5. Criterio de Desempeño y Ranking Multicriterio

In [24]:
df_parametros = pd.DataFrame(resultados)

# Criterio de selección biotecnológica: Producto entre velocidad y rendimiento
df_parametros['Desempeno_Score'] = df_parametros['mu'] * df_parametros['K']
df_parametros = df_parametros.sort_values(by='Desempeno_Score', ascending=False).reset_index(drop=True)

df_parametros['Categoria'] = 'Estándar'
df_parametros.loc[:2, 'Categoria'] = 'Top Destacado 🌟'

print("="*65)
print(f"{'RANKING DE AISLAMIENTOS SEGÚN DESEMPEÑO CINÉTICO':^65}")
print("="*65)
display(df_parametros[['Aislamiento', 'mu', 'K', 'R2', 'Categoria']])

        RANKING DE AISLAMIENTOS SEGÚN DESEMPEÑO CINÉTICO         


,Aislamiento,mu,K,R2,Categoria
0,Pleurotus_ostreatus_A1,0.2269,9.6020,0.9996,Top Destacado 🌟
1,Pycnoporus_sanguineus_P9,0.2165,9.7599,0.9998,Top Destacado 🌟
2,Schizophyllum_commune_S6,0.2583,7.9486,0.9995,Top Destacado 🌟
3,Hericium_erinaceus_H8,0.1685,11.6726,0.9991,Estándar
4,Lentinula_edodes_L4,0.2225,7.6525,0.9987,Estándar
5,Ganoderma_lucidum_G2,0.1734,9.1433,0.9991,Estándar
6,Trametes_versicolor_T3,0.1697,8.6142,0.9993,Estándar
7,Fomes_fomentarius_F10,0.1530,8.6700,0.9999,Estándar
8,Agaricus_bisporus_AB5,0.1205,7.9433,0.9994,Estándar
9,Grifola_frondosa_G7,0.0905,9.3109,0.9995,Estándar


### 6. Mapa de Decisión Biotecnológica y Exportación

In [25]:
fig_mapa = px.scatter(
    df_parametros,
    x='mu',
    y='K',
    color='Categoria',
    text='Aislamiento',
    size='R2',
    hover_data=['R2', 'Desempeno_Score'],
    title='<b>Mapa Cinético de Selección: Tasa (μ) vs Capacidad de Carga (K)</b>',
    labels={'mu': 'Tasa Específica de Crecimiento μ (h⁻¹)', 'K': 'Capacidad de Carga K (g/L)'},
    color_discrete_map={'Top Destacado 🌟': '#2ecc71', 'Estándar': '#95a5a6'}
)

fig_mapa.update_traces(textposition='top center', marker=dict(size=14))
fig_mapa.update_layout(template='plotly_white', title_font_size=16)
fig_mapa.show()

# Exportación del reporte final de resultados
resumen_path = DATA_DIR / "Resultados_Cineticos.xlsx"
df_parametros.to_excel(resumen_path, index=False)
print(f"\n✅ Reporte ejecutivo consolidado exportado con éxito en: {resumen_path}")


✅ Reporte ejecutivo consolidado exportado con éxito en: C:\Users\ivanc\Desktop\2. Inversiones\Python_Clases\fungi-kinetic-analytics\data\Resultados_Cineticos.xlsx


### 7. Conclusiones y Reporte Ejecutivo

# 🔬 Conclusiones y Evaluación Ejecutiva del Proyecto Cinético

A partir del análisis automatizado de las 10 cepas fúngicas evaluadas mediante el modelo logístico sigmoideo y la matriz de desempeño ($\mu \times K$), se desprenden las siguientes conclusiones técnicas:

### 1. Desempeño General de los Aislados
* **Ajuste de Modelos:** El coeficiente de determinación global ($R^2 > 0.98$ en promedio) demuestra que el modelo logístico se ajusta con alta precisión a las fases de crecimiento exponencial y estacionaria de los hongos evaluados.
* **Compromiso Cinético (Trade-off):** Se observa una divergencia natural entre las cepas con alta tasa de crecimiento específico ($\mu$) y aquellas que maximizan la capacidad de carga ($K$), evidenciando la necesidad de un índice ponderado para la selección industrial.

### 2. Identificación de Ceps "Super-Productoras" (`Top Destacado 🌟`)
* Las cepas clasificadas en el ranking superior combinan de manera óptima una velocidad de colonización rápida con rendimientos elevados de biomasa, lo que minimiza los tiempos de residencia en los biorreactores y optimiza la productividad volumétrica.
* Estos aislados seleccionados representan los candidatos ideales para escalar a fases de experimentación en medios líquidos o sólidos a mayor escala.

### 3. Próximos Pasos Recomendados
1. **Validación Experimental:** Contrastar los parámetros teóricos predichos por el modelo con réplicas biológicas reales en laboratorio.
2. **Optimización de Medios:** Evaluar el impacto de diferentes fuentes de carbono y nitrógeno sobre las cepas destacadas para maximizar aún más su capacidad de carga ($K$).
3. **Automatización del Pipeline:** Mantener este flujo modular (`src/utils.py` + Notebook interactivo) para incorporar de forma dinámica nuevos lotes de ensayos a medida que se expanda el estudio.